# 1. Initializations

## 1.1 General imports

In [ ]:
### general
import datetime
import pickle
from itertools import islice

### data management
import pandas as pd
import numpy as np

### machine learning (scikit-learn)
from statsmodels.tsa.seasonal import seasonal_decompose
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Lasso
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import root_mean_squared_error
from statsmodels.graphics.tsaplots import plot_pacf, plot_acf
from sklearn.decomposition import PCA
from sklearn.feature_selection import SelectFromModel, SelectKBest, f_regression, mutual_info_regression, RFE, RFECV
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.compose import make_column_transformer
from sklearn.metrics import mean_absolute_error, r2_score

### graphical
import matplotlib.pyplot as plt
# for jupyter notebook management
%matplotlib inline
import seaborn as sns


## 1.2 General dataframe functions

In [ ]:
import smartcheck.dataframe_common as dfc
import smartcheck.dataframe_project_specific as dfps

## 1.3 Specific preprocessing classes

In [ ]:
import smartcheck.preprocessing_project_specific as pps

# 2. Loading and Preprocessing

In [ ]:
df_cpt_raw = dfc.load_dataset_from_config('velo_comptage_ml_ready_data', sep=',', index_col=0)

if df_cpt_raw is not None and isinstance(df_cpt_raw, pd.DataFrame):
    df_cpt = df_cpt_raw.copy()

In [ ]:
df_cpt.info()

## 2.1 Preprocessing pipelines

In [ ]:
keep_cols = [
    "nom_du_site_de_comptage",
    "comptage_horaire",
    "date_et_heure_de_comptage",
    "orientation_compteur",
    "latitude",
    "longitude",
    "arrondissement",
    "jour_ferie",
    "vacances_scolaires",
    "temperature_2m_c",
    "rain_mm",
    "snowfall_cm",
    "weather_code_wmo_code",
    "elevation",
    "weather_code_wmo_code_category",
]

pipe_preproc = Pipeline([
    ("filter_columns", pps.ColumnFilterTransformer(columns_to_keep=keep_cols)),
    ("add_datetime_features", pps.DatetimePeriodicsTransformer(timestamp_col="date_et_heure_de_comptage")),
])

df_raw = pipe_preproc.fit_transform(df_cpt)
if df_raw is not None and isinstance(df_raw, pd.DataFrame):
    df = df_raw.copy()

In [ ]:
# Verification des distributions après preprocessing
df.info()
display(df.select_dtypes(include=np.number).describe())
display(df.select_dtypes(include='object').describe())

## 2.2 Column Transformers

In [ ]:
num_col = list(df.drop(columns='comptage_horaire').select_dtypes(include=np.number).columns)
cat_col = list(df.select_dtypes(include='object').columns)
s_scaler = StandardScaler()
ohe_enc = OneHotEncoder(handle_unknown='ignore')
tr_num_col = Pipeline(
    steps = [
        ('standardisation', s_scaler)
    ]
)
tr_cat_col = Pipeline(
    steps = [
        ('encoder', ohe_enc)
    ]
)
tr_columns = make_column_transformer( 
    (tr_num_col, num_col),
    (tr_cat_col, cat_col)
)

# 3. Regression modeling

## 3.1 Linear Regresion

In [ ]:
pipe_linear_regression = Pipeline(
    steps= [
        ('preprocessing_column_transformation', tr_columns), 
        ('linear_regression_model',LinearRegression())
    ]
)

In [ ]:
models_results = {}
# pas d'aggrégation, juste un regroupement par nom de site et orientation (utilisé ensuite dans la boucle for)
grouped = df.groupby(["nom_du_site_de_comptage", "orientation_compteur"])
for compteur_id, df_compteur in grouped:
    # tri chrono + pipeline prétraitement + split
    df_compteur = df_compteur.sort_values("date_et_heure_de_comptage_local")
    X_train, X_train_dates, X_test, X_test_dates, y_train, y_test = \
        dfps.train_test_split_time_aware(
            df_compteur,
            timestamp_cols=["date_et_heure_de_comptage_utc", "date_et_heure_de_comptage_local"],
            target_col="comptage_horaire"
        )
    # pipeline + fit
    model = pipe_linear_regression.fit(X_train, y_train)
    y_test_pred = model.predict(X_test)
    models_results[compteur_id] = [model, X_test_dates, y_test, y_test_pred]

print(models_results.keys())

In [ ]:
# Limitation du nombre de modèles prédictions a afficher graphiquement (4 premiers pour le moment)
nb_compteurs = 4
for compteur, model_results in islice(models_results.items(), nb_compteurs):
    model = model_results[0]
    X_test_dates = model_results[1]
    y_test = model_results[2]
    y_test_pred = model_results[3]    
    plt.figure(figsize=(12, 8))
    plt.plot(
        X_test_dates.date_et_heure_de_comptage_local, 
        y_test, label='Valeurs réelles'
    )
    plt.plot(
        X_test_dates.date_et_heure_de_comptage_local, 
        y_test_pred, 
        label='Prédictions', 
        linestyle='--'
    )
    plt.title(f'Prédictions de la régression linéaire pour le compteur {compteur}')
    plt.xlabel('Date')
    # limitation à une période de 15 jours
    plt.xlim([pd.to_datetime('2025-04-01'), pd.to_datetime('2025-04-16')])
    plt.ylabel('Comptage Horaire')
    plt.legend()
    plt.show();